# Lab 2: Simple Sorts and Stacks

**Z5007 Programming and Data Structures · Lab 2 · Thursday 22 October 2026**

**Today:** ICA 1 runs **2:00–2:45 pm**. Lab 2 runs **2:45–4:40 pm**.

This lab implements what we covered since Lab 1: bubble, selection and insertion sort (Fri, Mon), the stack ADT (Wed), and the two ways to build a stack: on an array and on a linked list (Thu).

| Part | Tier | Marks |
|---|---|---|
| A. Three simple sorts, with counters | Core | 25 |
| B. `ArrayStack`: a fixed array plus `_top`, doubling when full | Core | 20 |
| C. `LinkedStack`: a chain of nodes, the head is the top | Core | 20 |
| D. One bracket checker, two stacks | Build | 10 |
| E. Measure the sorts; inversions | Build | 15 |
| F. `MinStack`: `get_min()` in O(1) | Challenge | 10 |

Follow the numbered `TODO` comments in each cell, delete its `raise NotImplementedError` line, then run the test cell after it: it must print `passed`. Do not edit tests. You must be able to explain every line. Submit on Google Classroom by 11:59 pm today.

**How to work.** The instructions tell you *what* each piece must do and what it may cost, not *how* to write it. That part is yours: draw it on paper, decide, then code. If you are stuck for more than 10 minutes, the **Hints** section at the end has one hint per part. Use it; it costs no marks. Every idea in this lab was taught in this week's lectures.

## Part A (Core). Three simple sorts
Each function sorts the list **in place** and returns a pair `(comparisons, moves)`:
* `bubble_sort(a)`: compare neighbours, swap if `a[i] > a[i+1]`; stop early if a pass makes no swaps. Count each swap as 1 move.
* `selection_sort(a)`: find the minimum of the unsorted part, swap it to the front (only if needed). Count each swap as 1 move.
* `insertion_sort(a)`: count each shift (an item moving one place right) as 1 move.

The tests also check that bubble sort and insertion sort are **stable**. Decide which comparison guarantees it, and say why in a comment.

In [ ]:
def bubble_sort(a):
    comps = moves = 0
    # TODO 1: run passes. Each pass ends one position earlier, because the largest unsorted item
    #         "bubbles" to the end of the unsorted part and is then final.
    # TODO 2: in a pass, compare every pair of neighbours in the unsorted part (count each comparison);
    #         swap them if they are out of order (count each swap as a move).
    #         Which comparison keeps equal items in their original order (stable)?
    # TODO 3: if a whole pass makes no swap, the list is already sorted: stop early.
    raise NotImplementedError   # delete this line once your code is written
    return comps, moves

def selection_sort(a):
    comps = moves = 0
    # TODO 1: for each position i from the front (the last position fills itself):
    # TODO 2:     scan the unsorted part a[i ...] and remember the INDEX of its smallest item
    #             (count every comparison).
    # TODO 3:     swap that item into position i, but only if it is not already there (count the swap).
    raise NotImplementedError   # delete this line once your code is written
    return comps, moves

def insertion_sort(a):
    comps = moves = 0
    # TODO 1: grow a sorted prefix: for each item after the first, take it out as the "key"
    #         (save it first: shifting will overwrite its slot).
    # TODO 2: walk left through the sorted prefix; while the item there is larger than the key,
    #         shift it one place right (count each comparison and each shift).
    #         Stop at the front of the list or at the first item that is not larger.
    # TODO 3: drop the key into the gap that is left.
    raise NotImplementedError   # delete this line once your code is written
    return comps, moves


In [ ]:
# TEST A: do not edit. It checks your three sorts in three ways.
import random
random.seed(2)                                    # same "random" lists on every run, so results are repeatable

# 1. Correctness: sort 300 random lists with each function and compare with Python's own sorted()
for f in (bubble_sort, selection_sort, insertion_sort):
    for _ in range(300):
        a = [random.randint(0, 50) for _ in range(random.randint(0, 25))]   # a list of 0 to 25 numbers from 0 to 50
        b = a[:]; f(b)                            # copy a, then sort the copy in place with your function
        assert b == sorted(a), f"{f.__name__} did not sort correctly"

# 2. The counters: answers we worked out in the lectures
assert bubble_sort([1, 2, 3, 4, 5]) == (4, 0), "bubble sort should stop after one pass on sorted data"   # 4 comparisons, 0 swaps
assert selection_sort(list(range(10, 0, -1)))[0] == 45, "selection sort always makes n(n-1)/2 comparisons"  # 10 x 9 / 2 = 45
assert insertion_sort([4, 2, 5, 1, 3])[1] == 6, "insertion sort on [4,2,5,1,3] makes 6 shifts"           # Monday's exercise

# 3. Stability: an Item is compared ONLY by its key k; the tag lets us see the original order
class Item:
    def __init__(self, k, tag): self.k, self.tag = k, tag
    def __lt__(self, o): return self.k < o.k      # makes  a < b  compare keys
    def __gt__(self, o): return self.k > o.k      # makes  a > b  compare keys
for f in (bubble_sort, insertion_sort):           # selection sort is not stable, so it is not tested here
    items = [Item(2, "first"), Item(1, "x"), Item(2, "second")]   # two items share key 2
    f(items)
    # a stable sort keeps "first" before "second" because they were in that order
    assert [i.tag for i in items] == ["x", "first", "second"], f"{f.__name__} must be stable (use >, not >=)"
print("Part A passed")


## Part B (Core). `ArrayStack`: a fixed array plus one integer
Build the array stack from Thursday's lecture. **Do not use `append` or `pop` on the list**: the list `[None] * capacity` is only the raw block of slots.
* State: `self._data = [None] * capacity` and `self._top = -1` (the index of the top item; −1 means empty).
* `push(x)`: if the array is full, first call `self._resize(2 * len(self._data))`; then move `_top` up one slot and write `x` there.
* `pop()`: read the top, **clear its slot** (set it to `None`), move `_top` down, return the item.
* `peek()`: read the top without changing anything. `pop` and `peek` on an empty stack must `raise IndexError`.
* `is_empty()` and `__len__()` follow from `_top`.
* `_resize(cap)`: make a new block of `cap` slots, copy the items across, use the new block.

Every operation must be O(1), except `push` when it resizes (O(n), rare: amortized O(1)).

In [ ]:
class ArrayStack:
    """LIFO stack on an array that doubles when full."""
    def __init__(self, capacity=8):
        self._data = [None] * capacity     # the block of slots: never call append/pop on it
        self._top = -1                     # index of the top item; -1 means empty

    def is_empty(self):
        # TODO: what value of _top means "empty"?
        raise NotImplementedError   # delete this line once your code is written

    def __len__(self):
        # TODO: _top is an INDEX. How many items are stored when _top is 4?
        raise NotImplementedError   # delete this line once your code is written

    def _resize(self, cap):
        # TODO: make a new block of cap empty slots, copy the stored items into the same positions,
        #       then use the new block from now on.
        raise NotImplementedError   # delete this line once your code is written

    def push(self, x):
        # TODO 1: if there is no free slot above _top, double the capacity first (use _resize).
        # TODO 2: move _top up one slot, then write x there. (Which order: move then write, or write then move?)
        raise NotImplementedError   # delete this line once your code is written

    def pop(self):
        # TODO 1: an empty stack must raise IndexError.
        # TODO 2: read the top item, clear its slot (so the array does not keep the object alive),
        #         move _top down, and return the item.
        raise NotImplementedError   # delete this line once your code is written

    def peek(self):
        # TODO: like pop, but only read: change nothing. Empty must raise IndexError.
        raise NotImplementedError   # delete this line once your code is written


In [ ]:
# TEST B: do not edit
s = ArrayStack(2)                                 # a tiny capacity, so pushing 5 items must resize twice
assert s.is_empty() and len(s) == 0               # a new stack is empty
for x in [10, 20, 30, 40, 50]:
    s.push(x)
assert len(s) == 5 and s.peek() == 50             # 5 items; the last one pushed is on top
assert len(s._data) == 8, "capacity 2 should double to 4, then to 8"
assert s._top == 4, "_top is an index: 5 items means _top = 4"
assert s.pop() == 50 and s.pop() == 40 and len(s) == 3        # LIFO: last in, first out
assert s._data[3] is None and s._data[4] is None, "pop must clear the slot it empties"
assert s._data[:3] == [10, 20, 30]                # the remaining items are untouched, bottom first
for _ in range(3): s.pop()                        # empty the stack
for bad in (s.pop, s.peek):                       # popping or peeking an empty stack must be an error
    try:
        bad(); raise AssertionError("empty stack must raise IndexError")
    except IndexError:
        pass                                      # the right error was raised: good
# Thursday's exercise 1: capacity 4; push 5, push 8, pop, push 3, 7, 1, 9 (the 9 forces a doubling)
t = ArrayStack(4)
t.push(5); t.push(8); t.pop(); t.push(3); t.push(7); t.push(1); t.push(9)
assert t._data == [5, 3, 7, 1, 9, None, None, None] and t._top == 4
print("Part B passed")


## Part C (Core). `LinkedStack`: a chain of nodes; the head is the top
`Node` is given. Build `LinkedStack` with the **same five methods** as `ArrayStack` (`push`, `pop`, `peek`, `is_empty`, `__len__`), the same exceptions, and every operation O(1) in the worst case.
* State: `self._head` (the top node, or `None` when empty) and `self._size`.
* `push(x)`: create a node whose `next` is the old head, then make it the new head. Mind the order.
* `pop()`: read the head's data, move the head to the next node, return the data.

Then answer C.2 in the cell below the test.

In [ ]:
class Node:
    __slots__ = ("data", "next")
    def __init__(self, data, next=None):
        self.data = data      # the value
        self.next = next      # the node below (None at the bottom)

class LinkedStack:
    """LIFO stack on a singly linked list; the head is the top."""
    def __init__(self):
        self._head = None     # the top node; None means empty
        self._size = 0        # number of items

    def is_empty(self):
        # TODO: when is there no top node?
        raise NotImplementedError   # delete this line once your code is written

    def __len__(self):
        # TODO: return the item count you keep up to date in push and pop.
        raise NotImplementedError   # delete this line once your code is written

    def push(self, x):
        # TODO: create a node for x that points at the CURRENT top, then make it the new top; update the size.
        #       Order matters: what is lost if you move the head first?
        raise NotImplementedError   # delete this line once your code is written

    def pop(self):
        # TODO 1: an empty stack must raise IndexError.
        # TODO 2: read the top node's data, make the node below the new top, update the size, return the data.
        #         (Python frees the old node once nothing points to it.)
        raise NotImplementedError   # delete this line once your code is written

    def peek(self):
        # TODO: return the top node's data without changing anything. Empty must raise IndexError.
        raise NotImplementedError   # delete this line once your code is written


In [ ]:
# TEST C: do not edit
s = LinkedStack()
assert s.is_empty() and len(s) == 0               # a new stack is empty
for x in ["a", "b"]:
    s.push(x)
# the head must be the top: head -> "b" -> "a" -> None
assert s._head.data == "b" and s._head.next.data == "a" and s._head.next.next is None, "the head must be the top"
assert s.pop() == "b"                             # LIFO: "b" was pushed last
s.push("c")                                       # Thursday's exercise 2: now head -> "c" -> "a" -> None
assert s.peek() == "c" and len(s) == 2 and s._head.next.data == "a"
assert [s.pop(), s.pop()] == ["c", "a"] and s.is_empty() and s._head is None
for bad in (s.pop, s.peek):                       # popping or peeking an empty stack must be an error
    try:
        bad(); raise AssertionError("empty stack must raise IndexError")
    except IndexError:
        pass
big = LinkedStack()                               # 100,000 pushes: fast only if push is O(1)
for i in range(100_000): big.push(i)
assert len(big) == 100_000 and big.pop() == 99_999
print("Part C passed")


**C.2 (write below, 2–3 sentences).** Someone puts the top of a singly linked stack at the **tail** instead of the head. Which operation becomes O(n), and why does keeping a `tail` pointer not fix it?

*Your answer here.*

## Part D (Build). One bracket checker, two stacks
Write `balanced(text, make_stack)`: True if every `( [ {` is closed by the matching `) ] }` in the right order; ignore other characters. Create the stack with `s = make_stack()` and use **only** `push`, `pop`, `peek` and `is_empty`, never `_data` or `_head`.

The test runs your function with **both** `ArrayStack` and `LinkedStack`. If it passes with both, you have used the stack as an ADT: the code depends on what a stack promises, not on how it is built.

In [ ]:
def balanced(text, make_stack):
    pairs = {")": "(", "]": "[", "}": "{"}    # each closer -> the opener it needs
    s = make_stack()                           # use only push, pop, peek and is_empty on s
    # TODO 1: go through the text one character at a time; ignore anything that is not a bracket.
    # TODO 2: an opener goes on the stack.
    # TODO 3: a closer must match the most recent unclosed opener. Name the two ways this can fail.
    # TODO 4: at the end, what must be true of the stack for the text to be balanced?
    raise NotImplementedError   # delete this line once your code is written


In [ ]:
# TEST D: do not edit
cases = [("([{}])", True),               # properly nested
         ("", True),                     # nothing to match
         ("print(a[0] + f(x))", True),   # other characters are ignored
         ("([)]", False),                # ")" arrives while "[" is on top: wrong order
         ("((()", False),                # one "(" is never closed: leftover
         (")(", False),                  # a closer with an empty stack
         ("{[]}()", True),
         ("]", False)]
for S in (ArrayStack, LinkedStack):      # the SAME function must work with BOTH stacks
    for text, ok in cases:
        assert balanced(text, S) == ok, f"balanced({text!r}) with {S.__name__} should be {ok}"
print("Part D passed with both ArrayStack and LinkedStack")


## Part E (Build). Measure the sorts, and count inversions
1. Write `count_inversions(a)`: the number of pairs i < j with a[i] > a[j]. A simple O(n²) double loop is fine.
2. **E.2 Predict first (write it at the top of your E.3 answer, before you run it):** on nearly sorted data, which sort will be fastest and which will not improve at all? Then run the measurement cell. It times all three sorts on random and nearly sorted data of 2,000 items.

In [ ]:
def count_inversions(a):
    # An inversion is a pair of positions i < j with a[i] > a[j].
    # TODO: count every such pair. A simple O(n^2) double loop is fine (the inner loop starts after i).
    raise NotImplementedError   # delete this line once your code is written


In [ ]:
# TEST E and measurement: do not edit
import time, random
random.seed(3)
# 1. Insertion sort makes exactly one shift per inversion (Monday's lecture)
for _ in range(20):
    a = [random.randint(0, 100) for _ in range(30)]
    assert insertion_sort(a[:])[1] == count_inversions(a), "shifts must equal inversions"
# 2. Build two lists of 2,000 numbers: one random, one nearly sorted (sorted, then 10 random swaps)
n = 2000
rnd = [random.random() for _ in range(n)]
near = sorted(rnd)
for _ in range(10):
    i, j = random.randrange(n), random.randrange(n); near[i], near[j] = near[j], near[i]
# 3. Time each sort on a copy of each list and print a table
print(f"{'sort':<16}{'random (s)':>12}{'nearly sorted (s)':>20}")
for f in (bubble_sort, selection_sort, insertion_sort):
    t = []
    for data in (rnd, near):
        b = data[:]; t0 = time.perf_counter(); f(b); t.append(time.perf_counter() - t0)   # seconds taken
    print(f"{f.__name__:<16}{t[0]:>12.4f}{t[1]:>20.4f}")
print("Part E passed")


**E.3 (write below).** Which sort wins on nearly sorted data, and by roughly how much? Explain using inversions. Why does selection sort not benefit at all?

*Your answer here.*

## Part F (Challenge). `MinStack`: `get_min()` in O(1)
Extend `LinkedStack` with `get_min()`, returning the smallest item currently on the stack in **O(1)** (it must raise `IndexError` when empty). `push` and `pop` must stay O(1). This was Thursday's exercise 3: think about what each node could remember.

Then answer F.2 below the test.

In [ ]:
class MinStack(LinkedStack):
    # TODO: make get_min() O(1) while push and pop stay O(1).
    #       A single "current minimum" field is not enough: after a pop it cannot tell you the new minimum.
    #       Idea: let every node also remember the smallest value in itself and everything below it.
    #       Override push, pop and peek so that this extra information is stored and ignored correctly,
    #       then write get_min (empty must raise IndexError).
    def get_min(self):
        raise NotImplementedError   # delete this line once your code is written


In [ ]:
# TEST F: do not edit
s = MinStack()
for x, m in [(5, 5), (3, 3), (7, 3)]:             # Thursday's exercise 3: push 5, 3, 7; the minimum goes 5, 3, 3
    s.push(x); assert s.get_min() == m and s.peek() == x
assert s.pop() == 7 and s.get_min() == 3
assert s.pop() == 3 and s.get_min() == 5, "after popping 3 the minimum must go back to 5"
assert s.pop() == 5 and s.is_empty()
try:
    s.get_min(); raise AssertionError("get_min on an empty stack must raise IndexError")
except IndexError:
    pass
# 20,000 random pushes and pops, checked against a simple reference that tracks the minimum
import random, time
random.seed(4)
s, ref, mins = MinStack(), [], []
for _ in range(20_000):
    if ref and random.random() < 0.4:
        assert s.pop() == ref.pop(); mins.pop()
    else:
        v = random.randint(0, 10**6); s.push(v); ref.append(v); mins.append(v if not mins else min(v, mins[-1]))
    if ref: assert s.get_min() == mins[-1]
# speed: 200,000 get_min calls must be fast, which is only possible if get_min is O(1)
t = time.perf_counter()
for i in range(200_000, 0, -1): s.push(i)
for _ in range(200_000): s.get_min()
assert time.perf_counter() - t < 3, "too slow: is get_min really O(1)?"
print("Part F passed")


**F.2 (write below, 2–3 sentences).** Why is a single `self._min` field not enough? What goes wrong after a `pop`?

*Your answer here.*

## Hints (only if you are stuck for 10 minutes)
* **A.** Insertion sort: hold a[i] aside, move larger items right until you find its place. Selection sort: swap only when the minimum is not already in place. Use `>` (not `>=`) to stay stable.
* **B.** Full means `_top == len(self._data) - 1`. Push: resize if full, then `_top += 1`, then write. Pop: read, set the slot to `None`, then `_top -= 1`. Size is `_top + 1`.
* **C.** Push in one line: `self._head = Node(x, self._head)`. Pop: save `self._head.data`, then `self._head = self._head.next`. Empty means `self._head is None`.
* **D.** Push openers; on a closer, the stack must not be empty and the popped opener must be its partner; at the end the stack must be empty.
* **E.** Count pairs with two nested loops, `j` starting at `i + 1`.
* **F.** Store a pair `(value, min_so_far)` in each node; the new node's minimum is `min(x, minimum of the node below)`.

## Before you submit
* All test cells print `passed` (Part F only if attempted). Written answers C.2, E.3 (and F.2) complete.
* *Kernel → Restart & Run All* works. If you used an AI tool, add one line saying what for.